# Part 2: Text Classification on AG News Dataset 

### 2.1: Dataset: AG News

In [32]:
from datasets import load_dataset
import numpy as np

dataset = load_dataset("sh0416/ag_news")
print(dataset)

train_data = dataset['train']
test_data = dataset['test']
print(train_data[0])  
print(test_data[0])   

# Combine title + description into one text field
train_texts = [title + " " + desc for title, desc in zip(train_data['title'], train_data['description'])]
test_texts  = [title + " " + desc  for title, desc  in zip(test_data['title'],  test_data['description'])]


train_labels = np.array(train_data['label'])
test_labels  = np.array(test_data['label'])

print(f"Number of training samples: {len(train_texts)}")
print(f"Number of test samples: {len(test_texts)}")
print(f"Example training text (combined):\n{train_texts[0]}")
print(f"Example training label: {train_labels[0]}")
print(f"Unique labels: {np.unique(train_labels)}")           
print(f"Label mapping (typical): 1=World, 2=Sports, 3=Business, 4=Sci/Tech")

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})
{'label': 3, 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)', 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}
{'label': 3, 'title': 'Fears for T N pension after talks', 'description': "Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."}
Number of training samples: 120000
Number of test samples: 7600
Example training text (combined):
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Example training label: 3
Unique labels: [1 2 3 4]
Label mapping (typical): 1=World, 2=Sports, 3=Business, 4=Sci/Tech


### 2.2: Preprocessing for Classification

In [33]:
import re
import string

def preprocess_text(text: str) -> list[str]:

    # Lowercase
    text = text.lower()
    
    #replace punctuation with space, for example: "can't" will be "can t"
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    
    #remove whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    #tokenization
    tokens = text.split()
    
    return tokens

example_text = train_texts[0]
tokens = preprocess_text(example_text)

print(f"Original text: {example_text}")
print(f"\nAfter preprocessing (tokens): {tokens}")  
print(f"\nNumber of tokens: {len(tokens)}")


print("Preprocessing all training texts...")
train_tokens_list = [preprocess_text(text) for text in train_texts]

print("Preprocessing all test texts...")
test_tokens_list = [preprocess_text(text) for text in test_texts]

print("Preprocessing finished!")
print(f"First training document tokens (first 10 words): {train_tokens_list[0][:10]}")
print(f"First test document tokens (first 10 words): {test_tokens_list[0][:10]}")

Original text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

After preprocessing (tokens): ['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters', 'short', 'sellers', 'wall', 'street', 's', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again']

Number of tokens: 24
Preprocessing all training texts...
Preprocessing all test texts...
Preprocessing finished!
First training document tokens (first 10 words): ['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters']
First test document tokens (first 10 words): ['fears', 'for', 't', 'n', 'pension', 'after', 'talks', 'unions', 'representing', 'workers']


### 2.3: Naive Bayes Classification

**Class prior probability**

$$
P(c) = \frac{N_c}{N}
$$

Where:
- $N_c$ = number of documents in class $c$
- $N$ = total number of documents in the training set

**Likelihood (word probability given class) – with Laplace smoothing**

$$
P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha \cdot |V|}
$$

Where:
- $\text{count}(w, c)$ = number of times word $w$ appears in all documents of class $c$
- $\alpha$ = smoothing parameter (usually $\alpha = 1$ for Laplace smoothing)
- $|V|$ = vocabulary size (total number of unique words in the training set)
- $\sum_{w'} \text{count}(w', c)$ = total number of word tokens in class $c$

**Prediction – find the most probable class**

$$
\hat{c} = \arg\max_c \left( \log P(c) + \sum_{w \in d} \text{count}(w, d) \cdot \log P(w \mid c) \right)
$$

Where:
- $\hat{c}$ = predicted class for document $d$
- $\text{count}(w, d)$ = number of times word $w$ appears in document $d$
- The summation runs over all words that appear in the test document $d$
- Logarithms are used to avoid underflow and turn products into sums

#### 2.3.1: Implementation From Scratch

In [34]:
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple

def train_multinomial_naive_bayes(
    train_tokens: List[List[str]],          
    train_labels: np.ndarray,               
    alpha: float = 1.0                      
) -> Dict:
    
    # Get classes and map them to 0-based indices
    classes = np.unique(train_labels)                   
    n_classes = len(classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    n_docs = len(train_tokens)

    
    # Build vocabulary
    vocab_set = set()
    for tokens in train_tokens:
        vocab_set.update(tokens)
    
    vocab = sorted(list(vocab_set))
    vocab_size = len(vocab)
    word_to_idx = {w: i for i, w in enumerate(vocab)}

    print(f"Vocabulary size: {vocab_size:,}")


    # 2. Class priors: P(c) = (count(c) + 0) / N   
    class_counts = np.zeros(n_classes, dtype=np.float64)
    for lbl in train_labels:
        class_counts[class_to_idx[lbl]] += 1
    
    class_priors = class_counts / n_docs
    log_class_priors = np.log(class_priors + 1e-12)

    # 3. Word counts per class: count(w, c)
    count_matrix = np.zeros((n_classes, vocab_size), dtype=np.float64)
    
    for tokens, label in zip(train_tokens, train_labels):
        c_idx = class_to_idx[label]
        word_counter = Counter(tokens)
        for word, cnt in word_counter.items():
            if word in word_to_idx:                     
                widx = word_to_idx[word]
                count_matrix[c_idx, widx] += cnt

    # 4. Likelihood with Laplace smoothing
    #    P(w|c) = (count(w,c) + α) / (total_words_c + α * |V|)
    total_words_per_class = count_matrix.sum(axis=1)   
    total_words_per_class = np.maximum(total_words_per_class, 1e-10)
    
    likelihood = (count_matrix + alpha) / \
                 (total_words_per_class[:, np.newaxis] + alpha * vocab_size)
    
    log_likelihood = np.log(likelihood + 1e-12)

    # ───────────────────────────────────────────────
    # Package everything
    # ───────────────────────────────────────────────
    model = {
        'vocab': vocab,
        'word_to_idx': word_to_idx,
        'classes': classes,
        'log_class_priors': log_class_priors,           
        'log_likelihood': log_likelihood,               
        'alpha': alpha,
        'class_to_idx': class_to_idx
    }
    
    return model


def predict_naive_bayes(
    model: Dict,
    test_tokens_list: List[List[str]]
) -> np.ndarray:
    log_prior = model['log_class_priors']               
    log_like  = model['log_likelihood']                 
    vocab     = model['vocab']
    w2i       = model['word_to_idx']
    classes   = model['classes']
    
    n_classes = len(classes)
    predictions = []
    
    for tokens in test_tokens_list:
        # Score for each class: log P(c) + Σ log P(w|c) for w in document
        scores = np.full(n_classes, log_prior)          # start with log prior
        
        doc_counter = Counter(tokens)
        
        for word, count in doc_counter.items():
            if word in w2i:
                widx = w2i[word]
                # Add count * log P(w|c) for each class
                scores += count * log_like[:, widx]
        
        # Choose class with highest score
        best_idx = np.argmax(scores)
        pred_label = classes[best_idx]
        predictions.append(pred_label)
    
    return np.array(predictions)

In [35]:


print("Training Multinomial Naive Bayes...")
nb_model = train_multinomial_naive_bayes(
    train_tokens_list,
    train_labels,
    alpha=1.0
)

print("\nPredicting on test set...")
test_predictions = predict_naive_bayes(nb_model, test_tokens_list)

# Quick accuracy check (you can compare with test_labels)
accuracy = np.mean(test_predictions == test_labels)
print(f"Test accuracy (from-scratch NB):({accuracy*100:.2f}%)")

Training Multinomial Naive Bayes...
Vocabulary size: 65,015

Predicting on test set...
Test accuracy (from-scratch NB):(90.04%)


#### 2.3.2: Comparison with Scikit-learn

In [37]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline


# Create pipeline
nb_sklearn = Pipeline([
    ('vectorizer', CountVectorizer(
        lowercase=True,               # let it lowercase (matches our manual step)
        token_pattern=r'(?u)\b\w+\b', # simple word tokens
        stop_words=None,              # (matches our preprocessing)
        min_df=1                      # keep all terms
    )),
    ('classifier', MultinomialNB(alpha=1.0))   # same Laplace smoothing as from-scratch
])

print("Training scikit-learn pipeline on raw texts...")
nb_sklearn.fit(train_texts, train_labels)

print("Predicting on test set...")
sk_predictions = nb_sklearn.predict(test_texts)

#Evaluation 

sk_accuracy = accuracy_score(test_labels, sk_predictions)

print(f"\nscikit-learn MultinomialNB accuracy (raw texts): {sk_accuracy:.4f}  ({sk_accuracy*100:.2f}%)")
print(f"From-scratch NB accuracy (manual tokens):        (90.04%)")
print(f"Difference (sklearn raw - from-scratch):         {sk_accuracy - 0.9004:+.4f}")

# Detailed report
print("\nClassification Report (scikit-learn on raw texts):\n")
print(classification_report(test_labels, sk_predictions, 
                          target_names=['World', 'Sports', 'Business', 'Sci/Tech'],
                          digits=4))


Training scikit-learn pipeline on raw texts...
Predicting on test set...

scikit-learn MultinomialNB accuracy (raw texts): 0.9004  (90.04%)
From-scratch NB accuracy (manual tokens):        (90.04%)
Difference (sklearn raw - from-scratch):         -0.0000

Classification Report (scikit-learn on raw texts):

              precision    recall  f1-score   support

       World     0.9087    0.8958    0.9022      1900
      Sports     0.9490    0.9789    0.9637      1900
    Business     0.8786    0.8421    0.8600      1900
    Sci/Tech     0.8638    0.8847    0.8742      1900

    accuracy                         0.9004      7600
   macro avg     0.9000    0.9004    0.9000      7600
weighted avg     0.9000    0.9004    0.9000      7600

